## 6 Avaliação — o que é genuinamente anômalo?

Um detector global costuma marcar como “anomalia” tudo o que pertence ao regime menos comum. Para evitar isso, treinamos um **Isolation Forest separado para cada regime**.

Regras:
- o modelo de anomalias é treinado apenas com linhas que tenham no máximo **30% de ausência original**;
- a pontuação é calculada **dentro do regime**;
- cada regime usa seu próprio percentil **99%** como limiar;
- linhas de baixa qualidade são mantidas como categoria separada e **não geram alerta de anomalia**.

Assim, uma condição de alta intensidade pode ser perfeitamente normal se fizer parte de um regime recorrente; ela só é anômala se for extrema **em relação aos outros pontos daquele mesmo regime**.

In [ ]:
anomaly_score = np.full(len(Z), np.nan)
anomaly_flag = np.zeros(len(Z), dtype=bool)
thresholds = {}

for r in sorted(np.unique(regimes)):
    idx = np.where(regimes == r)[0]
    train_idx = idx[~low_quality[idx]]

    detector = IsolationForest(
        n_estimators=400,
        contamination='auto',
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    detector.fit(Z_cluster[train_idx])

    score_regime = -detector.score_samples(Z_cluster[idx])
    score_train = -detector.score_samples(Z_cluster[train_idx])
    threshold = float(np.quantile(score_train, 0.99))

    anomaly_score[idx] = score_regime
    anomaly_flag[idx] = score_regime >= threshold
    thresholds[int(r)] = threshold

anomaly_flag = anomaly_flag & ~low_quality

print(f'Candidatos a anomalia: {anomaly_flag.sum():,} ({100*anomaly_flag.mean():.2f}% do total)')
print(f'Baixa qualidade de dados: {low_quality.sum():,} ({100*low_quality.mean():.2f}% do total)')

In [ ]:
plt.figure(figsize=(9, 6))
plt.scatter(P[:, 0], P[:, 1], c=regimes, s=8, alpha=0.25)
plt.scatter(P[anomaly_flag, 0], P[anomaly_flag, 1], marker='x', s=45)
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('Anomalias condicionadas ao regime')
plt.tight_layout()
plt.show()

In [ ]:
anomaly_percentile = np.zeros(len(Z))
for r in sorted(np.unique(regimes)):
    idx = np.where(regimes == r)[0]
    order = np.argsort(anomaly_score[idx])
    ranks = np.empty(len(idx), dtype=float)
    ranks[order] = np.arange(1, len(idx) + 1) / len(idx)
    anomaly_percentile[idx] = ranks

anomaly_idx = np.where(anomaly_flag)[0]
top_idx = anomaly_idx[np.argsort(anomaly_percentile[anomaly_idx])[::-1]][:20]

important = ['CO(GT)', 'C6H6(GT)', 'NOx(GT)', 'NO2(GT)', 'PT08.S1(CO)', 'PT08.S5(O3)', 'T', 'RH']
important_idx = [list(feature_names).index(x) for x in important if x in feature_names]

html = ['<div style="overflow-x:auto"><table><tr><th>Data/hora</th><th>Regime</th><th>Percentil anomalia</th>']
html += [f'<th>{feature_names[j]}</th>' for j in important_idx]
html.append('</tr>')
for i in top_idx:
    html.append(f'<tr><td>{datetimes[i]}</td><td>{regimes[i]}</td><td>{100*anomaly_percentile[i]:.2f}%</td>')
    for j in important_idx:
        html.append(f'<td>{X_imputed[i, j]:.2f}</td>')
    html.append('</tr>')
html.append('</table></div>')
display(HTML(''.join(html)))

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(datetimes, anomaly_percentile, linewidth=0.7)
plt.scatter(datetimes[anomaly_flag], anomaly_percentile[anomaly_flag], marker='x', s=30)
plt.axhline(0.99, linestyle='--')
plt.ylabel('Percentil de anomalia no regime')
plt.xlabel('Tempo')
plt.title('Quando surgem comportamentos realmente atípicos?')
plt.tight_layout()
plt.show()